# Attention

_Deep Learning_

**Let the model focus on the most relevant input parts, with weights that add to 1.**

When producing each output, a model shouldn't treat all inputs equally. Some words matter more right now.

     Attention lets the model focus. It gives each input part a weight, and the weights add up to 1.

---

This notebook is a practice scaffold for the **Attention** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We'll compute scaled dot-product attention by hand and then check it against PyTorch's built-in multi-head version. We build it in three steps: (1) make queries/keys/values, (2) compute attention weights and blend the values, (3) compare with `nn.MultiheadAttention`.

### Step 1 — Create queries, keys, and values

Attention works over three sets of vectors. **Queries** ask "what am I looking for?", **keys** advertise "what do I contain?", and **values** carry the actual content. Here we make 4 queries and 6 key/value pairs, each an 8-dimensional vector.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# manual scaled dot-product attention
q = torch.randn(1, 4, 8)        # 4 queries, dim 8
k = torch.randn(1, 6, 8)        # 6 keys
v = torch.randn(1, 6, 8)        # 6 values

### Step 2 — Score, normalise, and blend

Each query is compared with every key by a dot product, then **scaled** by the square root of the dimension to keep the numbers stable. A softmax turns each row of scores into weights that sum to 1, and multiplying those weights by the values produces a weighted blend — the attention output.

In [ ]:
# Similarity of every query to every key, scaled by sqrt(dim).
scores = q @ k.transpose(-2, -1) / (8 ** 0.5)

# Softmax so each query's weights over the 6 keys sum to 1.
weights = F.softmax(scores, dim=-1)

# Weighted blend of the value vectors.
out = weights @ v

print("attention weights sum to 1:", weights.sum(-1).round().tolist())
print("manual out:", tuple(out.shape))           # (1, 4, 8)

### Step 3 — Compare with PyTorch's multi-head attention

PyTorch ships `nn.MultiheadAttention`, which runs several attention "heads" in parallel and combines them. Feeding it the same `q`, `k`, `v` confirms our manual version matches the shape the library produces.

In [ ]:
# the library version (multi-head)
mha = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)
attn_out, attn_w = mha(q, k, v)
print("multihead out:", tuple(attn_out.shape))

## Visualize the data & results

_Across the rows of a real digit image, which rows does self-attention make each row focus on?_

### Step 1 — Score every row of a real digit against every other

We take one real handwritten image (a digit 8) and treat each of its 8 pixel-rows as both a query and a key. A cosine-style similarity (dot product divided by the norms) measures how alike two rows are; multiplying by 4 sharpens the contrast before softmax.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
rows = digits.images[8].astype(float)    # 8 pixel-rows of a real 8

# Length of each row vector, used to normalise the similarity.
norm = np.linalg.norm(rows, axis=1)

# Cosine-like query-key similarity between every pair of rows, sharpened by 4.
scores = (rows @ rows.T) / (norm[:, None] * norm[None, :] + 1e-9) * 4

### Step 2 — Softmax each row into attention weights and plot

Applying a numerically-stable softmax to each row turns the similarities into attention weights that sum to 1 — so every row "distributes its attention" across the eight rows. The heatmap shows, for each row, which other rows it focuses on.

In [ ]:
# Numerically stable softmax per row -> attention weights summing to 1.
e = np.exp(scores - scores.max(axis=1, keepdims=True))
weights = e / e.sum(axis=1, keepdims=True)

plt.imshow(weights, cmap="viridis")
plt.xticks(range(8), [f"row{i+1}" for i in range(8)], rotation=45)
plt.yticks(range(8), [f"row{i+1}" for i in range(8)])
plt.title("Self-attention over the 8 rows of a real digit image (each row sums to 1)")
plt.colorbar()
plt.show()